In [95]:
import torch
import numpy as np
from einops import rearrange

In [108]:
x_items = 4
x_features = 4
embed_dim = 4
heads   = 2
output_dimension = 6


In [120]:
torch.manual_seed(7)
qkv = torch.nn.Linear(in_features=x_features, out_features=embed_dim*3, bias=False)
linear = torch.nn.Linear(in_features=embed_dim, out_features=output_dimension, bias=False)

x = torch.rand((1, x_items, x_features))
y = torch.rand([1, 4, 6])

In [121]:
q_weights = qkv.weight.mT[:, :4]
k_weights = qkv.weight.mT[:, 4:8]
v_weights = qkv.weight.mT[:, 8:]

In [122]:
# rank 0 qkv weights
qkv_rank0 = torch.zeros([4, 6])
qkv_rank0[:,:2] = q_weights[:,:2]
qkv_rank0[:,2:4] = k_weights[:,:2]
qkv_rank0[:,4:6] = v_weights[:,:2]

qkv_rank1 = torch.zeros([4, 6])
qkv_rank1[:,:2] = q_weights[:,2:]
qkv_rank1[:,2:4] = k_weights[:,2:]
qkv_rank1[:,4:6] = v_weights[:,2:]

In [123]:
qkv_map = qkv(x)
Q, K, V = torch.tensor_split(qkv_map, 3, dim=2)

Q1, K1, V1 = Q.clone(), K.clone(), V.clone()
Q1[:, :, 1], Q1[:, :, 2] = Q[:, :, 2], Q[:, :, 1]
K1[:, :, 1], K1[:, :, 2] = K[:, :, 2], K[:, :, 1]
V1[:, :, 1], V1[:, :, 2] = V[:, :, 2], V[:, :, 1]
Q = rearrange(Q1, 'n i (h d) -> n h i d', h = heads)
K = rearrange(K1, 'n i (h d) -> n h i d', h = heads)
V = rearrange(V1, 'n i (h d) -> n h i d', h = heads)

In [124]:
print(K)

tensor([[[[-0.1272,  0.3075],
          [ 0.0679,  0.3648],
          [ 0.1732,  0.2188],
          [-0.0189,  0.4483]],

         [[ 0.5644,  0.4213],
          [ 0.4258,  0.2531],
          [ 0.5210,  0.4106],
          [ 0.4392,  0.3541]]]], grad_fn=<PermuteBackward0>)


In [125]:
Q_scaled = Q * (embed_dim//heads) ** -0.5
attention = torch.matmul(Q_scaled, K.mT) 

attention_soft = torch.nn.Softmax(dim=-1)(attention)
print(f'with softmax:\n{attention_soft}')

with softmax:
tensor([[[[0.2447, 0.2522, 0.2512, 0.2519],
          [0.2475, 0.2521, 0.2432, 0.2572],
          [0.2463, 0.2513, 0.2522, 0.2502],
          [0.2440, 0.2533, 0.2453, 0.2573]],

         [[0.2429, 0.2597, 0.2452, 0.2521],
          [0.2413, 0.2609, 0.2445, 0.2532],
          [0.2426, 0.2591, 0.2454, 0.2529],
          [0.2432, 0.2589, 0.2456, 0.2524]]]], grad_fn=<SoftmaxBackward0>)


In [138]:
values = torch.matmul(attention_soft, V)

values_concat = rearrange(values, 'n h s d -> n s (h d)')
values_concat1 = values_concat.clone()

values_concat1[:,:,1],values_concat1[:,:,2] = values_concat[:,:,2], values_concat[:,:,1]
print(values_concat1)

tensor([[[ 0.5292, -0.4210,  0.2031,  0.2231],
         [ 0.5310, -0.4210,  0.2027,  0.2227],
         [ 0.5292, -0.4210,  0.2040,  0.2230],
         [ 0.5302, -0.4210,  0.2017,  0.2232]]], grad_fn=<CopySlices>)


In [143]:
linear.weight.mT[:3]

tensor([[ 0.2369,  0.4936, -0.2326, -0.0586, -0.1695,  0.1679],
        [-0.4669, -0.0297, -0.0010,  0.0550,  0.0196,  0.2878],
        [-0.4086, -0.3951,  0.2447,  0.1361, -0.2853,  0.0070]],
       grad_fn=<SliceBackward0>)

In [139]:
result = linear(values_concat1)
print(result)

tensor([[[ 0.3280,  0.1965, -0.0236, -0.1140, -0.2046, -0.0743],
         [ 0.3285,  0.1976, -0.0242, -0.1140, -0.2047, -0.0739],
         [ 0.3276,  0.1962, -0.0234, -0.1138, -0.2048, -0.0742],
         [ 0.3289,  0.1976, -0.0242, -0.1143, -0.2044, -0.0741]]],
       grad_fn=<UnsafeViewBackward0>)


In [128]:
loss = torch.nn.MSELoss()(result, y)
loss.backward()

In [136]:
result

tensor([[[ 0.3280,  0.1965, -0.0236, -0.1140, -0.2046, -0.0743],
         [ 0.3285,  0.1976, -0.0242, -0.1140, -0.2047, -0.0739],
         [ 0.3276,  0.1962, -0.0234, -0.1138, -0.2048, -0.0742],
         [ 0.3289,  0.1976, -0.0242, -0.1143, -0.2044, -0.0741]]],
       grad_fn=<UnsafeViewBackward0>)

In [ ]:
for param in attention_soft.parameters():
    if param.grad is not None:
        print(param.grad.mT)

AttributeError: 'Tensor' object has no attribute 'parameters'

In [98]:
print(y[:,:,3:])

tensor([[[0.2019, 0.7259, 0.4731],
         [0.8861, 0.7264, 0.8868],
         [0.5535, 0.4012, 0.1849],
         [0.7310, 0.3653, 0.8188]]])
